# 03 - Comandos DML em Tabelas Delta Lake

Demonstra operações DML (**INSERT**, **UPDATE**, **DELETE**) em tabelas Delta Lake armazenadas no MinIO, além de recursos como **HISTORY** e **TIME TRAVEL**.

**Pré-requisitos:** Notebook `02` executado (tabelas Delta no bucket `bronze`).

## 1. Configuração e SparkSession

In [ ]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

load_dotenv(override=True)

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')

spark = (
    SparkSession.builder
    .appName('DML_Delta_Lake')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada com sucesso!')

SparkSession criada com sucesso!


## 2. Registrar Tabelas Delta como SQL Tables

In [ ]:
tabelas_delta = ['ar_condicionado', 'clientes', 'vendedores', 'vendas']

for tabela in tabelas_delta:
    delta_path = f's3a://{BRONZE_BUCKET}/{tabela}'
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {tabela}
        USING delta
        LOCATION '{delta_path}'
    """)

# Listar tabelas registradas no Spark
print('Tabelas registradas no Spark:')
spark.sql('SHOW TABLES').show(truncate=False)

26/04/28 12:32:23 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Tabelas registradas no Spark:
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|default  |apolice  |false      |
|default  |carro    |false      |
|default  |cliente  |false      |
|default  |endereco |false      |
|default  |estado   |false      |
|default  |marca    |false      |
|default  |modelo   |false      |
|default  |municipio|false      |
|default  |regiao   |false      |
|default  |sinistro |false      |
|default  |telefone |false      |
+---------+---------+-----------+



## 3. Consultar Dados Atuais (SELECT)

In [ ]:
print('=== AR CONDICIONADOS ===')
spark.sql('SELECT * FROM ar_condicionado ORDER BY id').show()

print('=== CLIENTES ===')
spark.sql('SELECT * FROM clientes ORDER BY cd_cliente').show()

print('=== VENDEDORES ===')
spark.sql('SELECT * FROM vendedores ORDER BY cd_vendedor').show()

# %%
# Contagem de registros por tabela
print(f'{"Tabela":<20} {"Registros":>10}')
print('-' * 32)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<20} {count:>10}')

=== REGIÕES ===


26/04/28 12:32:35 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+------------+
|cd_regiao|   nm_regiao|
+---------+------------+
|        1|       NORTE|
|        2|    NORDESTE|
|        3|     SUDESTE|
|        4|         SUL|
|        5|CENTRO-OESTE|
+---------+------------+

=== MARCAS ===


+--------+----------+
|cd_marca|  nm_marca|
+--------+----------+
|       1| CHEVROLET|
|       2|VOLKSWAGEN|
|       3|      FIAT|
|       4|      FORD|
|       5|    TOYOTA|
|       6|     HONDA|
|       7|   RENAULT|
|       8|   HYUNDAI|
|       9|    NISSAN|
|      10|      JEEP|
+--------+----------+

=== MODELOS (primeiros 10) ===


+---------+--------+-----------+
|cd_modelo|cd_marca|  nm_modelo|
+---------+--------+-----------+
|        1|       1|       ONIX|
|        2|       1|     PRISMA|
|        3|       1|    TRACKER|
|        4|       1|       SPIN|
|        5|       1|      CRUZE|
|        6|       1|        S10|
|        7|       1|    EQUINOX|
|        8|       1|TRAILBLAZER|
|        9|       1|    MONTANA|
|       10|       1|     CAMARO|
+---------+--------+-----------+



In [ ]:
print(f'{"Tabela":<20} {"Registros":>10}')
print('-' * 32)
for tabela in tabelas_delta:
    count = spark.sql(f'SELECT COUNT(*) as cnt FROM {tabela}').collect()[0]['cnt']
    print(f'{tabela:<20} {count:>10}')

Tabela           Registros
---------------------------


regiao                   5


estado                  27


municipio             5570
marca                   10
modelo                 100


cliente              20010


endereco             20010


telefone             20010


carro                10002


apolice              10000


sinistro             10000


---
## 4. INSERT - Inserir Novos Registros

Vamos inserir novos registros nas tabelas `marca`, `modelo` e `cliente`.

In [ ]:
print('--- INSERT: Novos Aparelhos ---')
spark.sql("SELECT COUNT(*) as antes FROM ar_condicionado").show()

spark.sql("""
    INSERT INTO ar_condicionado VALUES
    (2025, true, 4, 'Midea', 'AirVolution', '12000 BTU', 2300.00),
    (2024, false, 5, 'TCL', 'Elite', '9000 BTU', 1600.00)
""")

spark.sql("SELECT * FROM ar_condicionado ORDER BY id").show()
print('2 novos aparelhos inseridos!')

--- INSERT: Novas marcas ---
+-----+
|antes|
+-----+
|   10|
+-----+



+--------+----------+
|cd_marca|  nm_marca|
+--------+----------+
|       1| CHEVROLET|
|       2|VOLKSWAGEN|
|       3|      FIAT|
|       4|      FORD|
|       5|    TOYOTA|
|       6|     HONDA|
|       7|   RENAULT|
|       8|   HYUNDAI|
|       9|    NISSAN|
|      10|      JEEP|
|      11|     TESLA|
|      12|       BYD|
|      13|       GWM|
+--------+----------+

3 novas marcas inseridas!


In [ ]:
print('--- INSERT: Novos Clientes ---')

spark.sql("""
    INSERT INTO clientes VALUES
    (103, 'Belo Horizonte', 'Carlos Mendes'),
    (104, 'Curitiba', 'Fernanda Lima')
""")

spark.sql("SELECT * FROM clientes WHERE cd_cliente >= 103 ORDER BY cd_cliente").show()
print('2 novos clientes inseridos!')

--- INSERT: Novos modelos ---


+---------+--------+---------+
|cd_modelo|cd_marca|nm_modelo|
+---------+--------+---------+
|      101|      11|  MODEL 3|
|      102|      11|  MODEL Y|
|      103|      12|  DOLPHIN|
|      104|      12|SONG PLUS|
|      105|      13| HAVAL H6|
+---------+--------+---------+

5 novos modelos inseridos!


In [ ]:
print('--- INSERT: Novo Vendedor ---')

spark.sql("""
    INSERT INTO vendedores VALUES
    (3, 'Pedro Gelo', 'Sul'),
    (4, 'Lucas Frio', 'Oeste')
""")

spark.sql("SELECT * FROM vendedores WHERE cd_vendedor >= 3").show()
print('2 novos vendedores inseridos!')

--- INSERT: Novo cliente ---


+----------+-------------------+-----------+----+-------------+
|cd_cliente|               nome|        cpf|sexo|dt_nascimento|
+----------+-------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA TESTE|12345678900|   M|   1990-05-15|
|     99002|  MARIA SOUZA TESTE|98765432100|   F|   1985-11-20|
+----------+-------------------+-----------+----+-------------+

2 novos clientes inseridos!


---
## 5. UPDATE - Atualizar Registros

Vamos atualizar registros existentes nas tabelas.

In [ ]:
print('--- UPDATE: Ajuste de Preço ---')
print('ANTES:')
spark.sql("SELECT id, nm_marca, nm_modelo, valor FROM ar_condicionado WHERE id = 4").show()

spark.sql("""
    UPDATE ar_condicionado 
    SET valor = 2150.00 
    WHERE id = 4
""")

print('DEPOIS:')
spark.sql("SELECT id, nm_marca, nm_modelo, valor FROM ar_condicionado WHERE id = 4").show()

--- UPDATE: Renomear marca ---
ANTES:


+--------+--------+
|cd_marca|nm_marca|
+--------+--------+
|      11|   TESLA|
+--------+--------+

DEPOIS:


+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|      11|TESLA MOTORS|
+--------+------------+



In [ ]:
print('--- UPDATE via DeltaTable API: Mudança de Cidade ---')
from pyspark.sql.functions import lit

print('ANTES:')
spark.sql("SELECT * FROM clientes WHERE cd_cliente = 103").show()

dt_clientes = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/clientes')

dt_clientes.update(
    condition="cd_cliente = 103",
    set={"cidade": lit("Uberlândia")}
)

print('DEPOIS:')
spark.sql("SELECT * FROM clientes WHERE cd_cliente = 103").show()

--- UPDATE: Atualizar cliente ---
ANTES:
+----------+-------------------+-----------+----+-------------+
|cd_cliente|               nome|        cpf|sexo|dt_nascimento|
+----------+-------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA TESTE|12345678900|   M|   1990-05-15|
+----------+-------------------+-----------+----+-------------+

DEPOIS:


+----------+--------------------+-----------+----+-------------+
|cd_cliente|                nome|        cpf|sexo|dt_nascimento|
+----------+--------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA ATU...|11122233344|   M|   1990-05-15|
+----------+--------------------+-----------+----+-------------+



---
## 6. DELETE - Remover Registros

Vamos deletar registros de tabelas Delta.

In [ ]:
print('--- DELETE: Remover Vendedor ---')
print('ANTES:')
spark.sql("SELECT * FROM vendedores ORDER BY cd_vendedor").show()

spark.sql("DELETE FROM vendedores WHERE cd_vendedor = 4")

print('DEPOIS:')
spark.sql("SELECT * FROM vendedores ORDER BY cd_vendedor").show()

--- DELETE: Remover marca GWM ---
ANTES:
+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|       1|   CHEVROLET|
|       2|  VOLKSWAGEN|
|       3|        FIAT|
|       4|        FORD|
|       5|      TOYOTA|
|       6|       HONDA|
|       7|     RENAULT|
|       8|     HYUNDAI|
|       9|      NISSAN|
|      10|        JEEP|
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
|      13|         GWM|
+--------+------------+

DEPOIS:


+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|       1|   CHEVROLET|
|       2|  VOLKSWAGEN|
|       3|        FIAT|
|       4|        FORD|
|       5|      TOYOTA|
|       6|       HONDA|
|       7|     RENAULT|
|       8|     HYUNDAI|
|       9|      NISSAN|
|      10|        JEEP|
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
+--------+------------+



In [ ]:
print('--- DELETE via DeltaTable API: Remover Aparelho TCL ---')
dt_ac = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/ar_condicionado')

print('Aparelhos ANTES:')
spark.sql("SELECT id, nm_marca, nm_modelo FROM ar_condicionado").show()

dt_ac.delete("id = 5")

print('Aparelhos DEPOIS:')
spark.sql("SELECT id, nm_marca, nm_modelo FROM ar_condicionado").show()

--- DELETE: Remover cliente teste ---
+----------+--------------------+-----------+----+-------------+
|cd_cliente|                nome|        cpf|sexo|dt_nascimento|
+----------+--------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA ATU...|11122233344|   M|   1990-05-15|
|     99002|   MARIA SOUZA TESTE|98765432100|   F|   1985-11-20|
+----------+--------------------+-----------+----+-------------+

Apos DELETE:


+----------+--------------------+-----------+----+-------------+
|cd_cliente|                nome|        cpf|sexo|dt_nascimento|
+----------+--------------------+-----------+----+-------------+
|     99001|JOAO DA SILVA ATU...|11122233344|   M|   1990-05-15|
+----------+--------------------+-----------+----+-------------+



---
## 7. HISTORY - Histórico de Versões Delta

O Delta Lake mantém um log de transações que permite visualizar o histórico completo de alterações.

In [ ]:
# Histórico da tabela de Ar Condicionado
print('=== HISTORICO DA TABELA AR CONDICIONADO ===')
dt_ac = DeltaTable.forPath(spark, f's3a://{BRONZE_BUCKET}/ar_condicionado')
dt_ac.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)

=== HISTORICO DA TABELA MARCA ===
+-------+-------------------+---------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                                                                            |
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
print('=== HISTORICO DA TABELA CLIENTES ===')
spark.sql('DESCRIBE HISTORY clientes').select('version', 'timestamp', 'operation').show(truncate=False)

=== HISTORICO DA TABELA CLIENTE ===
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|3      |2026-04-28 12:33:47|DELETE   |
|2      |2026-04-28 12:33:34|UPDATE   |
|1      |2026-04-28 12:33:23|WRITE    |
|0      |2026-04-28 12:30:42|WRITE    |
+-------+-------------------+---------+



---
## 8. TIME TRAVEL - Viagem no Tempo

O Delta Lake permite ler versões anteriores dos dados.

In [ ]:
# Versão atual da tabela ar_condicionado
print('=== AR CONDICIONADO - VERSAO ATUAL ===')
spark.sql('SELECT id, nm_marca, valor FROM ar_condicionado ORDER BY id').show()

=== MARCA - VERSAO ATUAL ===
+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|       1|   CHEVROLET|
|       2|  VOLKSWAGEN|
|       3|        FIAT|
|       4|        FORD|
|       5|      TOYOTA|
|       6|       HONDA|
|       7|     RENAULT|
|       8|     HYUNDAI|
|       9|      NISSAN|
|      10|        JEEP|
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
+--------+------------+



In [ ]:
# Time Travel - Ler versão 0 (estado original antes dos INSERTS e UPDATES)
print('=== AR CONDICIONADO - VERSAO 0 (original) ===')
df_v0 = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/ar_condicionado')
df_v0.select('id', 'nm_marca', 'valor').orderBy('id').show()

=== MARCA - VERSAO 0 (original) ===


+--------+----------+
|cd_marca|  nm_marca|
+--------+----------+
|       1| CHEVROLET|
|       2|VOLKSWAGEN|
|       3|      FIAT|
|       4|      FORD|
|       5|    TOYOTA|
|       6|     HONDA|
|       7|   RENAULT|
|       8|   HYUNDAI|
|       9|    NISSAN|
|      10|      JEEP|
+--------+----------+



In [ ]:
# Time Travel - Comparar versão original vs atual
print('=== COMPARACAO: Versão 0 vs Atual ===')
df_original = spark.read.format('delta').option('versionAsOf', 0).load(f's3a://{BRONZE_BUCKET}/ar_condicionado')
df_atual = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/ar_condicionado')

print(f'Versão 0: {df_original.count()} registros')
print(f'Atual:    {df_atual.count()} registros')

# Mostrar registros que existem na atual, mas sofreram modificações ou foram adicionados
print('\nRegistros ADICIONADOS ou MODIFICADOS:')
df_atual.subtract(df_original).select('id', 'nm_marca', 'valor').show()

=== COMPARACAO: Versao 0 vs Atual ===


Versao 0: 10 registros
Atual:    12 registros

Registros ADICIONADOS:


+--------+------------+
|cd_marca|    nm_marca|
+--------+------------+
|      11|TESLA MOTORS|
|      12|    BYD AUTO|
+--------+------------+



---
## 9. Resumo Final

In [ ]:
print('=' * 60)
print('RESUMO DAS OPERACOES DML REALIZADAS')
print('=' * 60)
print()
print('INSERT:')
print('  - 2 novos aparelhos (Midea e TCL)')
print('  - 2 novos clientes (Carlos e Fernanda)')
print('  - 2 novos vendedores (Pedro e Lucas)')
print()
print('UPDATE:')
print('  - Aparelho Midea: Preço ajustado para 2150.00 (via SQL)')
print('  - Cliente Carlos: Mudança de cidade para Uberlândia (via Delta API)')
print()
print('DELETE:')
print('  - Vendedor Lucas removido (via SQL)')
print('  - Aparelho TCL removido (via Delta API)')
print()
print('HISTORY e TIME TRAVEL:')
print('  - Consulta ao log de transações do Delta Lake')
print('  - Recuperação da Tabela "ar_condicionado" na versão 0 (antes de todas as mudanças)')
print('=' * 60)

RESUMO DAS OPERACOES DML REALIZADAS

INSERT:
  - 3 novas marcas (TESLA, BYD, GWM)
  - 5 novos modelos (MODEL 3, MODEL Y, DOLPHIN, etc.)
  - 2 novos clientes

UPDATE:
  - marca TESLA -> TESLA MOTORS (via SQL)
  - marca BYD -> BYD AUTO (via DeltaTable API)
  - cliente 99001 nome e CPF atualizados

DELETE:
  - marca GWM removida (via SQL)
  - cliente 99002 removido (via SQL)
  - modelo HAVAL H6 removido (via DeltaTable API)

HISTORY e TIME TRAVEL:
  - Historico completo de transacoes
  - Leitura de versoes anteriores (versionAsOf)
  - Comparacao entre versoes


In [ ]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
